In [9]:
import os
import time
import random
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

ARCHITECTURES = [
    "resnet18",
    "resnet34",
    "efficientnet_b3",
    "densenet121"
]
NUM_CLASSES = 4
CLASS_NAMES = [
    "Non-Demented",
    "Very Mild",
    "Mild",
    "Moderate Dementia"
]


STAGE1_EPOCHS = 4
STAGE2_EPOCHS = 5
STAGE1_LR = 1e-3
STAGE2_LR = 1e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 32


USE_CLASS_WEIGHTS = True

INFERENCE_WARMUP = 10
INFERENCE_REPEATS = 50

CHECKPOINT_DIR = "/kaggle/working"
WINNER_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best_alzheimer_cnn_screening_model.pth"
)

print("\n" + "=" * 60)
print("ALZHEIMER'S MRI CNN ARCHITECTURE SCREENING")
print("=" * 60)
print(f"Architectures     : {ARCHITECTURES}")
print(f"Number of classes : {NUM_CLASSES}")
print(f"Batch size        : {BATCH_SIZE}")
print(f"Class weighting   : {USE_CLASS_WEIGHTS}")
print(f"\nStage 1:")
print(f"  Epochs          : {STAGE1_EPOCHS}")
print(f"  Learning rate   : {STAGE1_LR}")
print("  Backbone        : Frozen")
print(f"\nStage 2:")
print(f"  Epochs          : {STAGE2_EPOCHS}")
print(f"  Learning rate   : {STAGE2_LR}")
print("  Backbone        : Unfrozen")
print(f"  Weight decay    : {WEIGHT_DECAY}")
print(f"\nTotal epochs/model: {STAGE1_EPOCHS + STAGE2_EPOCHS}")
print("=" * 60)

Using device: cuda
GPU: Tesla T4

ALZHEIMER'S MRI CNN ARCHITECTURE SCREENING
Architectures     : ['resnet18', 'resnet34', 'efficientnet_b3', 'densenet121']
Number of classes : 4
Batch size        : 32
Class weighting   : True

Stage 1:
  Epochs          : 4
  Learning rate   : 0.001
  Backbone        : Frozen

Stage 2:
  Epochs          : 5
  Learning rate   : 0.0001
  Backbone        : Unfrozen
  Weight decay    : 0.0001

Total epochs/model: 9


In [10]:


from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from PIL import Image

try:
    import imagehash
except ImportError:
    os.system("pip install imagehash --quiet")
    import imagehash

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png")
HAMMING_THRESHOLD = 5

AUGMENTED_DIR = "/kaggle/input/datasets/uraninjo/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset"
ORIGINAL_DIR = "/kaggle/input/datasets/uraninjo/augmented-alzheimer-mri-dataset/OriginalDataset"

FOLDER_TO_CLASS_NAME = {
    "MildDemented": "Mild",
    "ModerateDemented": "Moderate Dementia",
    "NonDemented": "Non-Demented",
    "VeryMildDemented": "Very Mild",
}



def build_dataframe(base_dir):
    records = []
    for root, dirs, files in os.walk(base_dir):
        for fname in files:
            if fname.lower().endswith(VALID_EXTENSIONS):
                filepath = os.path.join(root, fname)
                folder_class_name = os.path.basename(root)
                records.append({"filepath": filepath, "folder_class_name": folder_class_name})
    return pd.DataFrame(records)

df_augmented = build_dataframe(AUGMENTED_DIR)
df_original = build_dataframe(ORIGINAL_DIR)

for df_ in (df_augmented, df_original):
    missing = set(df_["folder_class_name"].unique()) - set(FOLDER_TO_CLASS_NAME.keys())
    assert not missing, f"Missing folder mapping for: {missing}"
    df_["class_name"] = df_["folder_class_name"].map(FOLDER_TO_CLASS_NAME)

class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
for df_ in (df_augmented, df_original):
    df_["label"] = df_["class_name"].map(class_to_idx)

print(f"Augmented pool (before filtering): {len(df_augmented)}")
print(f"Original pool: {len(df_original)}")
print("\nConfirmed class_name -> label index mapping:")
for name, idx in class_to_idx.items():
    print(f"  {idx}: {name}")



def remove_corrupt_files(df_):
    bad = []
    for p in df_["filepath"]:
        try:
            Image.open(p).verify()
        except Exception:
            bad.append(p)
    if bad:
        print(f"  Removing {len(bad)} unreadable files")
        df_ = df_[~df_["filepath"].isin(bad)].reset_index(drop=True)
    return df_

print("\nChecking augmented set for corrupt files...")
df_augmented = remove_corrupt_files(df_augmented)
print("Checking original set for corrupt files...")
df_original = remove_corrupt_files(df_original)



val_df, test_df = train_test_split(
    df_original, test_size=0.5, stratify=df_original["label"], random_state=SEED
)
eval_df = pd.concat([val_df, test_df], ignore_index=True)

print(f"\nVal: {len(val_df)} | Test: {len(test_df)}")
print("\nVal class distribution:")
print(val_df["class_name"].value_counts())
print("\nTest class distribution:")
print(test_df["class_name"].value_counts())



print("\nHashing eval (original) images...")
eval_df["phash"] = eval_df["filepath"].apply(
    lambda p: imagehash.phash(Image.open(p).convert("L"))
)

print("Hashing augmented pool (slow step, ~34k images — this will take a while)...")
df_augmented["phash"] = df_augmented["filepath"].apply(
    lambda p: imagehash.phash(Image.open(p).convert("L"))
)

eval_hashes = eval_df["phash"].dropna().tolist()

def is_near_duplicate_of_eval(train_hash, eval_hashes, threshold=HAMMING_THRESHOLD):
    if train_hash is None:
        return False
    for eh in eval_hashes:
        if (train_hash - eh) <= threshold:
            return True
    return False

print("Filtering augmented pool against eval set (may take several minutes)...")
df_augmented["is_leaked"] = df_augmented["phash"].apply(
    lambda h: is_near_duplicate_of_eval(h, eval_hashes)
)

num_leaked = df_augmented["is_leaked"].sum()
print(f"\nAugmented images flagged as near-duplicates of val/test: {num_leaked} "
      f"out of {len(df_augmented)} ({100 * num_leaked / len(df_augmented):.1f}%)")

train_df = df_augmented[~df_augmented["is_leaked"]].reset_index(drop=True)
print(f"Clean training set after filtering: {len(train_df)}")

print("\nTrain class distribution (post-filter):")
print(train_df["class_name"].value_counts())

if len(train_df) == 0:
    raise ValueError(
        "All augmented images were flagged as near-duplicates of val/test. "
        "The augmented folder cannot be used as-is — fall back to the "
        "OriginalDataset-only pipeline (Block 0 v3) instead."
    )

# ------------------------------------------------------------
# 5. Transforms
# ------------------------------------------------------------

IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

# ------------------------------------------------------------
# 6. Dataset class
# ------------------------------------------------------------

class MRIDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("L").convert("RGB")
        image = self.transform(image)
        return image, row["label"]

# ------------------------------------------------------------
# 7. WeightedRandomSampler — handles remaining class imbalance
#    in the filtered training pool.
# ------------------------------------------------------------

train_class_counts = train_df["label"].value_counts().sort_index()
sample_weights = train_df["label"].map(1.0 / train_class_counts).values

sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)



train_ds = MRIDataset(train_df, train_transform)
val_ds = MRIDataset(val_df, eval_transform)
test_ds = MRIDataset(test_df, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"\ntrain_loader: {len(train_loader)} batches (augmented, filtered, oversampled)")
print(f"val_loader  : {len(val_loader)} batches (original)")
print(f"test_loader : {len(test_loader)} batches (original)")
print("\nBlock 0 complete — train_loader, val_loader, test_loader are ready.")

Augmented pool (before filtering): 33984
Original pool: 6400

Confirmed class_name -> label index mapping:
  0: Non-Demented
  1: Very Mild
  2: Mild
  3: Moderate Dementia

Checking augmented set for corrupt files...
Checking original set for corrupt files...

Val: 3200 | Test: 3200

Val class distribution:
class_name
Non-Demented         1600
Very Mild            1120
Mild                  448
Moderate Dementia      32
Name: count, dtype: int64

Test class distribution:
class_name
Non-Demented         1600
Very Mild            1120
Mild                  448
Moderate Dementia      32
Name: count, dtype: int64

Hashing eval (original) images...
Hashing augmented pool (slow step, ~34k images — this will take a while)...
Filtering augmented pool against eval set (may take several minutes)...

Augmented images flagged as near-duplicates of val/test: 16442 out of 33984 (48.4%)
Clean training set after filtering: 17542

Train class distribution (post-filter):
class_name
Non-Demented        

In [11]:

CLASSIFIER_ATTR = {
    "resnet18": "fc",
    "resnet34": "fc",
    "efficientnet_b3": "classifier",
    "densenet121": "classifier",
}


def freeze_batchnorm_running_stats(model):

    for module in model.modules():
        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            module.eval()


def build_model(architecture_name, num_classes=NUM_CLASSES):
  

    architecture_name = architecture_name.lower()

    if architecture_name not in CLASSIFIER_ATTR:
        raise ValueError(
            f"Unknown architecture: {architecture_name}. "
            f"Choose from {ARCHITECTURES}"
        )

 

    if architecture_name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif architecture_name == "resnet34":
        model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif architecture_name == "efficientnet_b3":
        model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        # EfficientNet's classifier is Sequential(Dropout, Linear) — [-1] is the Linear
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)

    elif architecture_name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)



    for parameter in model.parameters():
        parameter.requires_grad = False

    classifier_module = getattr(model, CLASSIFIER_ATTR[architecture_name])
    for parameter in classifier_module.parameters():
        parameter.requires_grad = True



    model = model.to(DEVICE)

    

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(
        f"[{architecture_name}] total params: {total_params:,} | "
        f"trainable (Stage 1): {trainable_params:,} "
        f"({100 * trainable_params / total_params:.2f}%)"
    )

    return model

In [12]:
# ============================================================
# BLOCK 3 — REUSABLE TWO-STAGE TRAINING FUNCTION
# ============================================================

def set_trainable_layers(model, architecture_name, train_backbone=False):
 

    for parameter in model.parameters():
        parameter.requires_grad = False

    classifier = getattr(model, CLASSIFIER_ATTR[architecture_name])
    for parameter in classifier.parameters():
        parameter.requires_grad = True

    if train_backbone:
        for parameter in model.parameters():
            parameter.requires_grad = True


def compute_class_weights(train_loader, num_classes=NUM_CLASSES):
    
    labels = []
    for _, batch_labels in train_loader:
        labels.extend(batch_labels.numpy())

    class_counts = np.bincount(labels, minlength=num_classes)
    weights = 1.0 / np.maximum(class_counts, 1)  # avoid div-by-zero
    weights = weights / weights.sum() * num_classes  # normalize
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


# ============================================================
# VALIDATION FUNCTION
# ============================================================

def evaluate_model(model, data_loader, criterion):


    model.eval()

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

            predictions = torch.argmax(outputs, dim=1)
            all_predictions.extend(predictions.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    average_loss = running_loss / len(data_loader.dataset)
    accuracy = accuracy_score(all_targets, all_predictions)
    macro_f1 = f1_score(all_targets, all_predictions, average="macro")

    return average_loss, accuracy, macro_f1


# ============================================================
# MAIN REUSABLE TRAINING FUNCTION
# ============================================================

def train_model(
    model,
    architecture_name,
    train_loader,
    val_loader,
    epochs,
    learning_rate,
    train_backbone=False,
    stage_name="Stage",
    class_weights=None,
):
    

    # 1. Configure frozen / trainable layers
    set_trainable_layers(model, architecture_name, train_backbone=train_backbone)

    # 2. Display trainable parameter count
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{stage_name} | Trainable parameters: {trainable_params:,}")

    # 3. Loss function — wires in USE_CLASS_WEIGHTS from Block 1 config
    if USE_CLASS_WEIGHTS and class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    # 4. Optimizer
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY,
    )

    # 5. Training history
    history = {
        "train_loss": [], "train_accuracy": [], "train_macro_f1": [],
        "val_loss": [], "val_accuracy": [], "val_macro_f1": [],
    }

    # 6. Track best validation model — by macro-F1, not accuracy
    best_val_macro_f1 = -float("inf")
    best_model_state = copy.deepcopy(model.state_dict())

    # 7. Epoch loop
    for epoch in range(epochs):
        model.train()

       
        if not train_backbone:
            freeze_batchnorm_running_stats(model)

        running_loss = 0.0
        all_predictions = []
        all_targets = []

        for images, labels in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            predictions = torch.argmax(outputs, dim=1)
            all_predictions.extend(predictions.detach().cpu().numpy())
            all_targets.extend(labels.detach().cpu().numpy())

        train_loss = running_loss / len(train_loader.dataset)
        train_accuracy = accuracy_score(all_targets, all_predictions)
        train_macro_f1 = f1_score(all_targets, all_predictions, average="macro")

        val_loss, val_accuracy, val_macro_f1 = evaluate_model(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["train_macro_f1"].append(train_macro_f1)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)
        history["val_macro_f1"].append(val_macro_f1)

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_model_state = copy.deepcopy(model.state_dict())

        print(
            f"{stage_name} | Epoch [{epoch + 1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.4f} | Train F1: {train_macro_f1:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f} | Val F1: {val_macro_f1:.4f}"
        )

    # 8. Restore best validation state (by macro-F1)
    model.load_state_dict(best_model_state)

    return model, history

In [13]:
# ============================================================
# BLOCK 4 — METRICS & INFERENCE BENCHMARKING
# ============================================================

def calculate_total_parameters(model):
    """Calculate the total number of parameters in a model."""
    return sum(parameter.numel() for parameter in model.parameters())


# ============================================================
# PREDICTIONS
# ============================================================

def get_predictions(model, data_loader):

    model.eval()

    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            all_targets.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    return np.array(all_targets), np.array(all_predictions)


# ============================================================
# INFERENCE TIME BENCHMARK
# ============================================================

def benchmark_inference_time(
    model,
    data_loader,
    fixed_batch_size=BATCH_SIZE,
    warmup=INFERENCE_WARMUP,
    repeats=INFERENCE_REPEATS,
):

    model.eval()

    # Build a synthetic batch of the exact target size, using the
    # real data's shape/dtype as a template, rather than trusting
    # whatever batch DataLoader happens to yield first.
    sample_images, _ = next(iter(data_loader))
    single_image = sample_images[0:1]  # keep shape, drop to 1 sample
    images = single_image.repeat(fixed_batch_size, 1, 1, 1).to(DEVICE, non_blocking=True)

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)

    # Warm-up passes — not timed, lets CUDA kernels/cuDNN autotuning settle
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(images)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

    # Timed forward passes
    times = []
    with torch.no_grad():
        for _ in range(repeats):
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            start_time = time.perf_counter()
            _ = model(images)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            end_time = time.perf_counter()

            times.append(end_time - start_time)

    average_time_ms = np.mean(times) * 1000

    peak_memory_mb = None
    if DEVICE.type == "cuda":
        peak_memory_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2)

    # NOTE: this function leaves the model in .eval() mode as a side
    # effect. That's fine within this screening pipeline because
    # train_model() calls model.train() at the top of every epoch —
    # but if calculate_metrics() is ever called standalone between
    # Stage 1 and Stage 2 training, don't remove that train() call
    # thinking it's redundant.

    return average_time_ms, peak_memory_mb


# ============================================================
# COMPLETE METRICS FUNCTION
# ============================================================

def calculate_metrics(model, data_loader, class_names=CLASS_NAMES):


    # 1. Predictions
    targets, predictions = get_predictions(model, data_loader)

    # 2. Accuracy
    accuracy = accuracy_score(targets, predictions)

    # 3. Macro F1
    macro_f1 = f1_score(targets, predictions, average="macro", zero_division=0)

    # 4. Per-class recall — asserts label-order assumption instead of
    #    silently trusting it, since a label-encoding mismatch here
    #    would print correct-looking numbers under the wrong class names.
    assert len(class_names) == len(np.unique(targets)) or len(class_names) == NUM_CLASSES, (
        "class_names length doesn't match NUM_CLASSES — check CLASS_NAMES "
        "ordering against your dataset's actual label encoding before trusting "
        "the per-class recall breakdown below."
    )

    per_class_recall = recall_score(
        targets, predictions, average=None,
        labels=np.arange(len(class_names)), zero_division=0
    )
    recall_dict = {
        class_name: float(recall_value)
        for class_name, recall_value in zip(class_names, per_class_recall)
    }

    # 5. Moderate Dementia recall
    moderate_class_name = "Moderate Dementia"
    moderate_recall = recall_dict.get(moderate_class_name, np.nan)

    # 6. Total parameter count
    total_parameters = calculate_total_parameters(model)

    # 7. Inference speed + memory, on a fixed batch size
    inference_time_ms, peak_memory_mb = benchmark_inference_time(model, data_loader)

    # 8. Build final metrics dictionary
    metrics = {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "recall_non_demented": recall_dict["Non-Demented"],
        "recall_very_mild": recall_dict["Very Mild"],
        "recall_mild": recall_dict["Mild"],
        "recall_moderate": moderate_recall,
        "total_parameters": int(total_parameters),
        "inference_time_ms": float(inference_time_ms),
        "peak_memory_mb": peak_memory_mb,
        "per_class_recall": recall_dict,
    }

    # 9. Print results
    print("\n" + "-" * 60)
    print("MODEL EVALUATION")
    print("-" * 60)
    print(f"Accuracy              : {accuracy:.4f}")
    print(f"Macro-F1              : {macro_f1:.4f}")
    print("\nPer-class Recall:")
    for class_name, recall_value in recall_dict.items():
        print(f"  {class_name:20s}: {recall_value:.4f}")
    print(f"\nModerate Dementia Recall: {moderate_recall:.4f}")
    print(f"\nTotal Parameters      : {total_parameters:,}")
    print(f"Avg Inference / Batch  : {inference_time_ms:.3f} ms (batch size {BATCH_SIZE})")
    if peak_memory_mb is not None:
        print(f"Peak GPU Memory        : {peak_memory_mb:.1f} MB")
    print("-" * 60)

    return metrics

In [14]:
# ============================================================
# BLOCK 5 — ARCHITECTURE SCREENING LOOP
# ============================================================

import gc



screening_results = {}


trained_model_states = {}


training_histories = {}


class_weights = compute_class_weights(train_loader) if USE_CLASS_WEIGHTS else None
print(f"Class weights (if enabled): {class_weights}")


# ============================================================
# LOOP OVER ALL ARCHITECTURES
# ============================================================

for architecture_name in ARCHITECTURES:

    print("\n")
    print("=" * 80)
    print(f"STARTING ARCHITECTURE: {architecture_name.upper()}")
    print("=" * 80)

    architecture_start_time = time.perf_counter()

    try:

        model = build_model(
            architecture_name=architecture_name,
            num_classes=NUM_CLASSES
        )

        # ----------------------------------------------------
        # 2. Stage 1 — Train classifier head only
        # ----------------------------------------------------

        print("\n" + "-" * 70)
        print("STAGE 1 — CLASSIFIER HEAD TRAINING")
        print("-" * 70)

        model, stage1_history = train_model(
            model=model,
            architecture_name=architecture_name,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=STAGE1_EPOCHS,
            learning_rate=STAGE1_LR,
            train_backbone=False,
            stage_name=f"{architecture_name} | Stage 1",
            class_weights=class_weights,
        )

        # ----------------------------------------------------
        # 3. Stage 2 — Fine-tune entire network
        # ----------------------------------------------------

        print("\n" + "-" * 70)
        print("STAGE 2 — FULL NETWORK FINE-TUNING")
        print("-" * 70)

        model, stage2_history = train_model(
            model=model,
            architecture_name=architecture_name,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=STAGE2_EPOCHS,
            learning_rate=STAGE2_LR,
            train_backbone=True,
            stage_name=f"{architecture_name} | Stage 2",
            class_weights=class_weights,
        )

        # ----------------------------------------------------
        # 4. Evaluate final trained model on TEST set
        # ----------------------------------------------------

        print("\n" + "-" * 70)
        print(f"FINAL TEST EVALUATION — {architecture_name}")
        print("-" * 70)

        metrics = calculate_metrics(
            model=model,
            data_loader=test_loader,
            class_names=CLASS_NAMES
        )

        # ----------------------------------------------------
        # 5. Add architecture name + total training time to metrics
        # ----------------------------------------------------

        architecture_elapsed_minutes = (time.perf_counter() - architecture_start_time) / 60
        metrics["architecture"] = architecture_name
        metrics["training_time_minutes"] = round(architecture_elapsed_minutes, 2)

        # ----------------------------------------------------
        # 6. Store results
        # ----------------------------------------------------

        screening_results[architecture_name] = metrics

        # ----------------------------------------------------
        # 7. Store model state dict on CPU (not the live GPU model)
        # ----------------------------------------------------

        trained_model_states[architecture_name] = {
            k: v.cpu() for k, v in model.state_dict().items()
        }

        # ----------------------------------------------------
        # 8. Store training histories
        # ----------------------------------------------------

        training_histories[architecture_name] = {
            "stage1": stage1_history,
            "stage2": stage2_history
        }

        # ----------------------------------------------------
        # 9. Print completion message
        # ----------------------------------------------------

        print("\n" + "=" * 80)
        print(f"COMPLETED: {architecture_name.upper()} "
              f"({architecture_elapsed_minutes:.1f} min)")
        print("=" * 80)

    except RuntimeError as e:
        # Catches CUDA OOM and similar — logs the failure and moves on
        # to the next architecture instead of losing all prior results.
        print(f"\n[FAILED] {architecture_name} raised an error: {e}")
        print(f"Skipping {architecture_name} and continuing with remaining architectures.\n")
        screening_results[architecture_name] = {"architecture": architecture_name, "error": str(e)}

    finally:
        # ----------------------------------------------------
        # 10. Free GPU memory before the next architecture,
        #     whether this one succeeded or failed.
        # ----------------------------------------------------
        if "model" in dir():
            del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


# ============================================================
# SCREENING COMPLETE
# ============================================================

print("\n\n")
print("=" * 80)
print("ALL ARCHITECTURES COMPLETED")
print("=" * 80)

successful = [name for name, m in screening_results.items() if "error" not in m]
failed = [name for name, m in screening_results.items() if "error" in m]

print(f"Architectures evaluated successfully: {len(successful)}")
for architecture_name in successful:
    print(f"  ✓ {architecture_name}")

if failed:
    print(f"\nArchitectures that failed: {len(failed)}")
    for architecture_name in failed:
        print(f"  ✗ {architecture_name}: {screening_results[architecture_name]['error']}")

Class weights (if enabled): tensor([1.0113, 0.9973, 0.9854, 1.0060], device='cuda:0')


STARTING ARCHITECTURE: RESNET18
[resnet18] total params: 11,178,564 | trainable (Stage 1): 2,052 (0.02%)

----------------------------------------------------------------------
STAGE 1 — CLASSIFIER HEAD TRAINING
----------------------------------------------------------------------

resnet18 | Stage 1 | Trainable parameters: 2,052
resnet18 | Stage 1 | Epoch [1/4] | Train Loss: 1.0955 | Train Acc: 0.5297 | Train F1: 0.5190 | Val Loss: 0.9760 | Val Acc: 0.5600 | Val F1: 0.4667
resnet18 | Stage 1 | Epoch [2/4] | Train Loss: 0.9275 | Train Acc: 0.6111 | Train F1: 0.6022 | Val Loss: 0.9077 | Val Acc: 0.5663 | Val F1: 0.4674
resnet18 | Stage 1 | Epoch [3/4] | Train Loss: 0.8676 | Train Acc: 0.6359 | Train F1: 0.6272 | Val Loss: 0.8679 | Val Acc: 0.6156 | Val F1: 0.5240
resnet18 | Stage 1 | Epoch [4/4] | Train Loss: 0.8315 | Train Acc: 0.6517 | Train F1: 0.6422 | Val Loss: 0.8412 | Val Acc: 0.6181 | Val F1